# EduVision_DV — Module 2: Data Cleaning & Transformation

This notebook takes the **Module 1 integrated dataset** `university_raw_data.csv` and produces the final **Tableau-ready** `university_cleaned.csv`.

## Module 2 tasks
1. Remove duplicates
2. Standardize university names
3. Standardize country names
4. Normalize ranking metrics
5. Handle and validate missing values
6. Create a Tableau-ready dataset

## Final dataset design
The final dataset keeps one common `University_Name` and one common `Country`.

The following redundant source-identity columns are **not** included in the final cleaned file:
- `QS_University_Name`
- `THE_University_Name`
- `QS_Country`
- `THE_Country`

QS/THE performance metrics are retained because they are different analytical measures.

> **Missing-value policy:** unavailable source-specific numeric values are represented by `-1` in the Tableau-ready file, and unavailable text values by `Not Available`. This avoids inventing rankings or scores. `QS_Data_Available` and `THE_Data_Available` indicate whether the corresponding ranking source exists for the university.


In [1]:
# =========================================================
# 0. IMPORT LIBRARIES AND SET INPUT/OUTPUT PATHS
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

# The notebook should be inside:
# Milestone1/Module_2_Data Cleaning & Transformation/notebooks/

BASE = Path.cwd()

# If Jupyter starts in the notebooks folder
if BASE.name == "notebooks":
    MODULE_2 = BASE.parent
else:
    MODULE_2 = BASE

# Module 1 output
INPUT = (
    MODULE_2.parent
    / "Module_1_Data_Collection"
    / "processed_data"
    / "university_raw_data.csv"
)

# Module 2 output
OUTPUT_DIR = MODULE_2 / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = OUTPUT_DIR / "university_cleaned.csv"

print("Input file:")
print(INPUT)

print("\nOutput file:")
print(OUTPUT)

if not INPUT.exists():
    raise FileNotFoundError(
        "\nCould not find university_raw_data.csv at:\n"
        f"{INPUT}\n\n"
        "Check that the Module 1 output is located at "
        "Module_1_Data_Collection/processed_data/university_raw_data.csv"
    )

df = pd.read_csv(INPUT)

print("\nDataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))


Input file:
C:\Users\A\Desktop\EduVision_DV\Milestone1\Module_1_Data_Collection\processed_data\university_raw_data.csv

Output file:
C:\Users\A\Desktop\EduVision_DV\Milestone1\Module_2_Data Cleaning & Transformation\processed_data\university_cleaned.csv

Dataset loaded successfully.
Rows: 2883
Columns: 29


## 1. Inspect the Module 1 raw/integrated dataset

In [2]:
# Basic inspection

print("Dataset shape:", df.shape)

print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\nDuplicate complete rows:", df.duplicated().sum())

print("\nMissing values before cleaning:")
missing_before = df.isna().sum().sort_values(ascending=False)
print(missing_before[missing_before > 0].head(20))

print("\nYears found in integrated data:")
print(sorted(pd.to_numeric(df["Year"], errors="coerce").dropna().unique().tolist()))


Dataset shape: (2883, 29)

Column names:
1. QS_Rank
2. QS_Previous_Rank
3. QS_University_Name
4. QS_Country
5. QS_Region
6. QS_Size
7. QS_Focus
8. QS_Research_Level
9. QS_Status
10. QS_Academic_Reputation_Score
11. THE_Rank
12. THE_University_Name
13. THE_Country
14. THE_Student_Population
15. THE_Students_to_Staff_Ratio
16. THE_International_Students
17. THE_Female_to_Male_Ratio
18. THE_Overall_Score
19. THE_Teaching
20. THE_Research_Environment
21. THE_Research_Quality
22. THE_Industry_Impact
23. THE_International_Outlook
24. THE_Year
25. University_Name
26. Country
27. Year
28. Data_Source
29. QS_THE_Match_Status

Duplicate complete rows: 0

Missing values before cleaning:
QS_Previous_Rank                1494
QS_Status                       1429
QS_Size                         1383
QS_Research_Level               1383
QS_Rank                         1382
QS_Region                       1382
QS_Country                      1382
QS_University_Name              1382
QS_Focus           

## 2. Keep only 2026 data

Module 2 is based on the 2026 QS and THE datasets. The integrated file therefore remains restricted to analysis year **2026**.

For THE records, `THE_Year` is also checked when a value is present.


In [3]:
# Convert year fields to numeric
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

if "THE_Year" in df.columns:
    df["THE_Year"] = pd.to_numeric(df["THE_Year"], errors="coerce")

# Keep only analysis year 2026
df = df[df["Year"] == 2026].copy()

# If THE year exists, keep THE records only when their year is 2026.
# QS-only rows can have THE_Year missing and are retained.
if "THE_Year" in df.columns:
    invalid_the = df["THE_Year"].notna() & (df["THE_Year"] != 2026)
    df = df[~invalid_the].copy()

print("Rows after 2026 filtering:", len(df))
print("Analysis year(s):", sorted(df["Year"].dropna().unique().tolist()))


Rows after 2026 filtering: 2883
Analysis year(s): [2026]


## 3. Standardize university and country names

In [4]:
# =========================================================
# 3. STANDARDIZE UNIVERSITY AND COUNTRY NAMES
# =========================================================

def clean_text(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)

    return value if value else pd.NA


def normalize_university_key(value):
    # This key is used only for matching/deduplication.
    # The final University_Name remains readable.
    value = clean_text(value)

    if pd.isna(value):
        return pd.NA

    text = unicodedata.normalize(
        "NFKD", str(value)
    ).encode("ascii", "ignore").decode("ascii").lower()

    text = text.replace("&", " and ")
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\bthe\b", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def standardize_country(value):
    value = clean_text(value)

    if pd.isna(value):
        return pd.NA

    aliases = {
        "united states of america": "United States",
        "united states": "United States",
        "usa": "United States",
        "us": "United States",

        "united kingdom": "United Kingdom",
        "uk": "United Kingdom",
        "england": "United Kingdom",

        "russian federation": "Russia",
        "republic of korea": "South Korea",
        "south korea": "South Korea",
    }

    return aliases.get(
        str(value).lower(),
        str(value).strip()
    )


# Clean only the text columns that are actually needed.
text_columns = [
    "QS_University_Name",
    "THE_University_Name",
    "QS_Country",
    "THE_Country",
    "University_Name",
    "Country",
]

for column in text_columns:
    if column in df.columns:
        df[column] = df[column].map(clean_text)


# Create ONE final university name.
df["University_Name"] = (
    df["University_Name"]
    .fillna(df["QS_University_Name"])
    .fillna(df["THE_University_Name"])
    .map(clean_text)
)


# Create ONE final standardized country.
df["Country"] = (
    df["Country"]
    .fillna(df["QS_Country"])
    .fillna(df["THE_Country"])
    .map(standardize_country)
)


print("University names standardized.")
print("Country names standardized.")

print("\nSample standardized university names:")
print(df["University_Name"].head(10).to_string(index=False))

print("\nSample standardized countries:")
print(df["Country"].head(10).to_string(index=False))


University names standardized.
Country names standardized.

Sample standardized university names:
                            Aalborg University
                              Aalto University
                             Aarhus University
                 Abdelmalek Essaâdi University
             Abdul Wali Khan University Mardan
                       Abdullah Gül University
                        Aberystwyth University
                        Abo Akademi University
                          Abu Dhabi University
Abylkas Saginov Karaganda Technical University

Sample standardized countries:
             Denmark
             Finland
             Denmark
             Morocco
            Pakistan
              Turkey
      United Kingdom
             Finland
United Arab Emirates
          Kazakhstan


## 4. Normalize ranking, score, percentage and ratio metrics

QS ranking data contains both exact ranks and rank bands such as `741-750` or `1401+`.

For the Tableau-ready numeric `QS_Rank` and `QS_Previous_Rank`:
- exact rank → same number
- rank band → midpoint of the band
- `1401+` → 1401 as the lower-bound numeric value

This gives Tableau a consistent numeric ranking field without pretending that a rank band is an exact rank.


In [5]:
# =========================================================
# 4. NORMALIZE RANKING METRICS
# =========================================================

def normalize_rank(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().replace(",", "")

    # Example: 1401+
    plus_match = re.fullmatch(r"(\d+)\+", text)
    if plus_match:
        return float(plus_match.group(1))

    # Example: 741-750
    range_match = re.fullmatch(r"(\d+)\s*-\s*(\d+)", text)
    if range_match:
        low = float(range_match.group(1))
        high = float(range_match.group(2))
        return (low + high) / 2

    # Exact numeric value
    number_match = re.search(r"\d+(?:\.\d+)?", text)
    if number_match:
        return float(number_match.group())

    return np.nan


def numeric_clean(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.extract(r"(-?\d+(?:\.\d+)?)", expand=False),
        errors="coerce"
    )


# QS ranking fields
for column in ["QS_Rank", "QS_Previous_Rank"]:
    if column in df.columns:
        df[column] = df[column].map(normalize_rank)


# Other numeric QS/THE fields
numeric_columns = [
    "QS_Academic_Reputation_Score",
    "THE_Rank",
    "THE_Student_Population",
    "THE_Students_to_Staff_Ratio",
    "THE_Overall_Score",
    "THE_Teaching",
    "THE_Research_Environment",
    "THE_Research_Quality",
    "THE_Industry_Impact",
    "THE_International_Outlook",
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = numeric_clean(df[column])


# International students: 13% -> 13
if "THE_International_Students" in df.columns:
    df["THE_International_Students_Pct"] = pd.to_numeric(
        df["THE_International_Students"]
        .astype("string")
        .str.replace("%", "", regex=False)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )


# Female:Male ratio: 39 : 61 -> 39 and 61
if "THE_Female_to_Male_Ratio" in df.columns:
    ratio = df["THE_Female_to_Male_Ratio"].astype("string").str.extract(
        r"^\s*(\d+(?:\.\d+)?)\s*:\s*(\d+(?:\.\d+)?)\s*$"
    )

    df["THE_Female_Ratio"] = pd.to_numeric(
        ratio[0], errors="coerce"
    )

    df["THE_Male_Ratio"] = pd.to_numeric(
        ratio[1], errors="coerce"
    )


# Ensure analysis year is integer
df["Year"] = 2026

print("Ranking and score metrics normalized.")
print("QS Rank dtype:", df["QS_Rank"].dtype)
print("THE Rank dtype:", df["THE_Rank"].dtype)


Ranking and score metrics normalized.
QS Rank dtype: float64
THE Rank dtype: Float64


## 5. Remove duplicate university-country-year records and create source flags

In [6]:
# =========================================================
# 5. REMOVE DUPLICATES
# =========================================================

before_duplicates = len(df)

# Create a normalized matching key.
df["_University_Key"] = df["University_Name"].map(
    normalize_university_key
)

# Remove duplicate university-country-year combinations.
df = df.drop_duplicates(
    subset=["_University_Key", "Country", "Year"],
    keep="first"
).copy()

duplicates_removed = before_duplicates - len(df)

print("Records before duplicate removal:", before_duplicates)
print("Duplicates removed:", duplicates_removed)
print("Records after duplicate removal:", len(df))


# =========================================================
# SOURCE AVAILABILITY FLAGS
# =========================================================

df["QS_Data_Available"] = df["QS_Rank"].notna()
df["THE_Data_Available"] = df["THE_Rank"].notna()

print("\nQS records available:", int(df["QS_Data_Available"].sum()))
print("THE records available:", int(df["THE_Data_Available"].sum()))


Records before duplicate removal: 2883
Duplicates removed: 35
Records after duplicate removal: 2848

QS records available: 1501
THE records available: 2156


## 6. Create the final Tableau-ready dataset

The final dataset intentionally removes redundant source identity columns:

- `QS_University_Name`
- `THE_University_Name`
- `QS_Country`
- `THE_Country`

Only the unified fields `University_Name` and `Country` are retained.

QS/THE performance indicators remain because they represent different measurements.


In [7]:
# =========================================================
# 6. FINAL TABLEAU-READY COLUMNS
# =========================================================

final_columns = [
    # Common dimensions
    "University_Name",
    "Country",
    "Year",

    # QS metrics
    "QS_Rank",
    "QS_Previous_Rank",
    "QS_Academic_Reputation_Score",
    "QS_Region",
    "QS_Size",
    "QS_Focus",
    "QS_Research_Level",
    "QS_Status",

    # THE metrics
    "THE_Rank",
    "THE_Student_Population",
    "THE_Students_to_Staff_Ratio",
    "THE_International_Students_Pct",
    "THE_Female_Ratio",
    "THE_Male_Ratio",
    "THE_Overall_Score",
    "THE_Teaching",
    "THE_Research_Environment",
    "THE_Research_Quality",
    "THE_Industry_Impact",
    "THE_International_Outlook",

    # Integration fields
    "Data_Source",
    "QS_THE_Match_Status",

    # Tableau source-availability flags
    "QS_Data_Available",
    "THE_Data_Available",
]

# Keep only columns that exist in the source.
final_columns = [
    column for column in final_columns
    if column in df.columns
]

clean = df[final_columns].copy()

print("Final Tableau-ready columns:", len(clean.columns))

for i, column in enumerate(clean.columns, 1):
    print(f"{i}. {column}")


Final Tableau-ready columns: 27
1. University_Name
2. Country
3. Year
4. QS_Rank
5. QS_Previous_Rank
6. QS_Academic_Reputation_Score
7. QS_Region
8. QS_Size
9. QS_Focus
10. QS_Research_Level
11. QS_Status
12. THE_Rank
13. THE_Student_Population
14. THE_Students_to_Staff_Ratio
15. THE_International_Students_Pct
16. THE_Female_Ratio
17. THE_Male_Ratio
18. THE_Overall_Score
19. THE_Teaching
20. THE_Research_Environment
21. THE_Research_Quality
22. THE_Industry_Impact
23. THE_International_Outlook
24. Data_Source
25. QS_THE_Match_Status
26. QS_Data_Available
27. THE_Data_Available


## 7. Handle remaining missing values

We do **not** invent ranking scores.

Some universities are QS-only or THE-only, so the other ranking's metrics can legitimately be unavailable.

For the final Tableau-ready CSV:
- text unavailable → `Not Available`
- numeric unavailable → `-1`

The availability flags must be used when calculating QS/THE averages so that `-1` is never treated as a real score or rank.


In [8]:
# =========================================================
# 7. HANDLE MISSING VALUES
# =========================================================

# Text fields
text_columns = clean.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:
    clean[column] = (
        clean[column]
        .fillna("Not Available")
        .astype(str)
        .str.strip()
    )


# Numeric fields
numeric_columns = clean.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_columns = [
    column for column in numeric_columns
    if column != "Year"
]

for column in numeric_columns:
    clean[column] = pd.to_numeric(
        clean[column],
        errors="coerce"
    ).fillna(-1)


# Year
clean["Year"] = (
    pd.to_numeric(
        clean["Year"],
        errors="coerce"
    )
    .fillna(2026)
    .astype(int)
)


print("Missing-value handling completed.")
print("Actual remaining NaN cells:", int(clean.isna().sum().sum()))


Missing-value handling completed.
Actual remaining NaN cells: 0


## 8. Validate Module 2 evaluation requirements

In [9]:
# =========================================================
# 8. VALIDATION
# =========================================================

# Final missing values
missing_cells = int(clean.isna().sum().sum())
total_cells = clean.shape[0] * clean.shape[1]
missing_percentage = (
    missing_cells / total_cells * 100
    if total_cells
    else 0
)

# Duplicate validation
duplicate_keys = clean.duplicated(
    subset=["University_Name", "Country", "Year"]
).sum()

# Year validation
years = sorted(clean["Year"].unique().tolist())

# Ranking type validation
qs_rank_numeric = pd.api.types.is_numeric_dtype(clean["QS_Rank"])
the_rank_numeric = pd.api.types.is_numeric_dtype(clean["THE_Rank"])

# Ranking range checks
qs_rank_min = clean.loc[clean["QS_Rank"] >= 0, "QS_Rank"].min()
qs_rank_max = clean.loc[clean["QS_Rank"] >= 0, "QS_Rank"].max()

the_rank_min = clean.loc[clean["THE_Rank"] >= 0, "THE_Rank"].min()
the_rank_max = clean.loc[clean["THE_Rank"] >= 0, "THE_Rank"].max()


print("=" * 60)
print("MODULE 2 VALIDATION")
print("=" * 60)

print(f"Rows: {len(clean):,}")
print(f"Columns: {len(clean.columns)}")
print(f"Duplicates removed: {duplicates_removed:,}")
print(f"Remaining duplicate university-country-year records: {duplicate_keys}")
print(f"Remaining actual NaN cells: {missing_cells}")
print(f"Final missing percentage: {missing_percentage:.2f}%")
print(f"Year(s): {years}")

print("\nRanking consistency:")
print(f"QS_Rank numeric: {qs_rank_numeric}")
print(f"THE_Rank numeric: {the_rank_numeric}")
print(f"QS_Rank range: {qs_rank_min} to {qs_rank_max}")
print(f"THE_Rank range: {the_rank_min} to {the_rank_max}")

# Project checks
assert missing_percentage < 2, (
    "Missing-value requirement failed: "
    f"{missing_percentage:.2f}%"
)

assert duplicate_keys == 0, (
    "Duplicate university-country-year records remain."
)

assert years == [2026], (
    f"Unexpected years found: {years}"
)

assert qs_rank_numeric and the_rank_numeric, (
    "Ranking columns are not numeric."
)

print("\nALL MODULE 2 VALIDATION CHECKS PASSED.")


MODULE 2 VALIDATION
Rows: 2,848
Columns: 27
Duplicates removed: 35
Remaining duplicate university-country-year records: 0
Remaining actual NaN cells: 0
Final missing percentage: 0.00%
Year(s): [2026]

Ranking consistency:
QS_Rank numeric: True
THE_Rank numeric: True
QS_Rank range: 1.0 to 1401.0
THE_Rank range: 1.0 to 2191.0

ALL MODULE 2 VALIDATION CHECKS PASSED.


## 9. Save `university_cleaned.csv`

This is the official Module 2 dataset that should be used for **KPI Engineering (Module 3)** and later Tableau dashboard development.


In [10]:
# =========================================================
# 9. SAVE FINAL CLEANED DATASET
# =========================================================

clean.to_csv(
    OUTPUT,
    index=False
)

print("Successfully saved:")
print(OUTPUT)

print("\nFinal dataset shape:")
print(clean.shape)

print("\nFinal columns:")
print(clean.columns.tolist())


Successfully saved:
C:\Users\A\Desktop\EduVision_DV\Milestone1\Module_2_Data Cleaning & Transformation\processed_data\university_cleaned.csv

Final dataset shape:
(2848, 27)

Final columns:
['University_Name', 'Country', 'Year', 'QS_Rank', 'QS_Previous_Rank', 'QS_Academic_Reputation_Score', 'QS_Region', 'QS_Size', 'QS_Focus', 'QS_Research_Level', 'QS_Status', 'THE_Rank', 'THE_Student_Population', 'THE_Students_to_Staff_Ratio', 'THE_International_Students_Pct', 'THE_Female_Ratio', 'THE_Male_Ratio', 'THE_Overall_Score', 'THE_Teaching', 'THE_Research_Environment', 'THE_Research_Quality', 'THE_Industry_Impact', 'THE_International_Outlook', 'Data_Source', 'QS_THE_Match_Status', 'QS_Data_Available', 'THE_Data_Available']


## Tableau usage note

For QS calculations:

```text
QS_Data_Available = True
```

For THE calculations:

```text
THE_Data_Available = True
```

Do not include `-1` in averages or ranking calculations. `-1` represents an unavailable source-specific value, not a real ranking.

### Module 2 deliverables

- `university_cleaned.csv`
- `education_cleaning.ipynb`
